In [1]:
import pandas as pd

# read in relevant tables
people = pd.read_csv("EHRShot/sampled_person.csv")
visits = pd.read_csv("EHRShot/sampled_condition_occurrence.csv")
concepts = pd.read_csv("EHRShot/concept.csv")

C:\Users\patri\AppData\Local\Temp\ipykernel_26712\2330753205.py:5: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  visits = pd.read_csv("EHRShot/sampled_condition_occurrence.csv")
C:\Users\patri\AppData\Local\Temp\ipykernel_26712\2330753205.py:6: DtypeWarning: Columns (0,9) have mixed types. Specify dtype option on import or set low_memory=False.
  concepts = pd.read_csv("EHRShot/concept.csv")


In [2]:
# canonicalize id columns
people_ids = ['person_id', 'gender_concept_id', 'race_concept_id', 'ethnicity_concept_id']
people[people_ids] = people[people_ids].apply(pd.to_numeric, errors='coerce')
people.dtypes

concepts['concept_id'] = pd.to_numeric(concepts['concept_id'], errors='coerce')
concepts.dtypes

condition_ids = ['condition_concept_id', 'person_id', 'condition_occurrence_id']
visits[condition_ids] = visits[condition_ids].apply(pd.to_numeric, errors='coerce')
visits.dtypes

condition_occurrence_id            int64
person_id                          int64
condition_concept_id               int64
condition_start_DATE              object
condition_start_DATETIME          object
condition_end_DATE                object
condition_end_DATETIME            object
condition_type_concept_id          int64
stop_reason                      float64
provider_id                      float64
visit_occurrence_id              float64
visit_detail_id                  float64
condition_source_value            object
condition_source_concept_id        int64
condition_status_source_value     object
condition_status_concept_id        int64
trace_id                         float64
unit_id                          float64
load_table_id                     object
dtype: object

# 1. Prepare set of person nodes

In [3]:
people.describe() # 2000 entries

,person_id,gender_concept_id,year_of_birth,month_of_birth,day_of_birth,race_concept_id,ethnicity_concept_id,location_id,provider_id,care_site_id,person_source_value,gender_source_concept_id,race_source_concept_id,ethnicity_source_concept_id,trace_id,unit_id
count,2.000000e+03,2000.00000,2000.000000,2000.000000,2000.000000,2000.00000,2.000000e+03,1.997000e+03,1.790000e+03,454.000000,0.0,2000.00000,2000.0,2.000000e+03,0.0,0.0
mean,1.159705e+08,8519.56250,1960.011500,6.546000,15.833000,6478.97450,3.547633e+07,8.953843e+06,6.784153e+06,527974.041850,NaN,8519.56250,0.0,3.547633e+07,NaN,NaN
std,1.919994e+03,12.50297,18.826832,3.487544,8.788158,3641.79283,9.471111e+06,1.438006e+06,3.479738e+04,1106.967805,NaN,12.50297,0.0,9.471111e+06,NaN,NaN
min,1.159671e+08,8507.00000,1911.000000,1.000000,1.000000,0.00000,0.000000e+00,6.546561e+06,6.696393e+06,526564.000000,NaN,8507.00000,0.0,0.000000e+00,NaN,NaN
25%,1.159688e+08,8507.00000,1946.000000,3.000000,8.000000,8515.00000,3.800356e+07,7.682868e+06,6.757719e+06,527061.000000,NaN,8507.00000,0.0,3.800356e+07,NaN,NaN
50%,1.159705e+08,8532.00000,1957.000000,7.000000,16.000000,8527.00000,3.800356e+07,8.913501e+06,6.777141e+06,527448.000000,NaN,8532.00000,0.0,3.800356e+07,NaN,NaN
75%,1.159721e+08,8532.00000,1976.000000,10.000000,23.000000,8527.00000,3.800356e+07,1.017585e+07,6.811991e+06,528623.000000,NaN,8532.00000,0.0,3.800356e+07,NaN,NaN
max,1.159738e+08,8532.00000,2002.000000,12.000000,31.000000,8657.00000,3.800356e+07,1.148758e+07,6.900406e+06,530154.000000,NaN,8532.00000,0.0,3.800356e+07,NaN,NaN


1) Person nodes will be all entries in people, add attributes: race, ethnicity, gender, location, birth year, and "person" one hot
2) Filter to people who had a condition occurrence (visit)
3) Get unique conditions from visits, label with concept name, count of unique occurrences

In [4]:
len(set(people['person_id']).intersection(set(visits['person_id']))) # only 1842 people occur in visits
# filter to these people
people_filter = people[people['person_id'].isin(visits['person_id'])]
people_filter.describe() # now 1842 entries

,person_id,gender_concept_id,year_of_birth,month_of_birth,day_of_birth,race_concept_id,ethnicity_concept_id,location_id,provider_id,care_site_id,person_source_value,gender_source_concept_id,race_source_concept_id,ethnicity_source_concept_id,trace_id,unit_id
count,1.842000e+03,1842.000000,1842.000000,1842.000000,1842.000000,1842.000000,1.842000e+03,1.840000e+03,1.780000e+03,452.000000,0.0,1842.000000,1842.0,1.842000e+03,0.0,0.0
mean,1.159705e+08,8519.744300,1961.066232,6.557546,15.796417,6580.988599,3.620861e+07,8.937167e+06,6.784068e+06,527976.955752,NaN,8519.744300,0.0,3.620861e+07,NaN,NaN
std,1.920127e+03,12.501006,18.342447,3.506524,8.785347,3577.546623,8.064005e+06,1.436051e+06,3.470534e+04,1108.235724,NaN,12.501006,0.0,8.064005e+06,NaN,NaN
min,1.159671e+08,8507.000000,1920.000000,1.000000,1.000000,0.000000,0.000000e+00,6.546561e+06,6.696393e+06,526564.000000,NaN,8507.000000,0.0,0.000000e+00,NaN,NaN
25%,1.159688e+08,8507.000000,1947.000000,3.000000,8.000000,8515.000000,3.800356e+07,7.679590e+06,6.757719e+06,527061.000000,NaN,8507.000000,0.0,3.800356e+07,NaN,NaN
50%,1.159705e+08,8532.000000,1958.000000,7.000000,16.000000,8527.000000,3.800356e+07,8.893778e+06,6.777141e+06,527448.000000,NaN,8532.000000,0.0,3.800356e+07,NaN,NaN
75%,1.159721e+08,8532.000000,1977.000000,10.000000,23.000000,8527.000000,3.800356e+07,1.014352e+07,6.811878e+06,528623.000000,NaN,8532.000000,0.0,3.800356e+07,NaN,NaN
max,1.159738e+08,8532.000000,2002.000000,12.000000,31.000000,8657.000000,3.800356e+07,1.148561e+07,6.900406e+06,530154.000000,NaN,8532.000000,0.0,3.800356e+07,NaN,NaN


In [5]:
# select desired people columns
person_nodes = people_filter[['person_id', 'gender_concept_id', 'year_of_birth', 'race_concept_id', 'ethnicity_concept_id']].copy()
# replace concept ids with concept names
concept_map = concepts[['concept_id', 'concept_name']].drop_duplicates().set_index('concept_id')['concept_name']
person_nodes['gender'] = person_nodes['gender_concept_id'].map(concept_map)
person_nodes['race'] = person_nodes['race_concept_id'].map(concept_map)
person_nodes['ethnicity'] = person_nodes['ethnicity_concept_id'].map(concept_map)
# drop concept id columns
person_nodes = person_nodes.drop(columns=['gender_concept_id', 'race_concept_id', 'ethnicity_concept_id']).reset_index(drop=True)
person_nodes.head()

,person_id,year_of_birth,gender,race,ethnicity
0,115967179,1953,FEMALE,No matching concept,No matching concept
1,115967279,1969,MALE,Asian,Not Hispanic or Latino
2,115969631,1981,MALE,Asian,Not Hispanic or Latino
3,115972932,1982,MALE,No matching concept,Not Hispanic or Latino
4,115968415,1951,MALE,White,Not Hispanic or Latino


In [6]:
# NOTE BIASES IN DATA: skewed toward white non-hispanic
person_nodes['race'].value_counts()

race
White                                        1008
No matching concept                           420
Asian                                         307
Black or African American                      79
Native Hawaiian or Other Pacific Islander      23
American Indian or Alaska Native                5
Name: count, dtype: int64

In [7]:
# one-hot encode gender, race, ethnicity
ohe_cols = ['gender', 'race', 'ethnicity']
person_nodes_ohe = pd.get_dummies(person_nodes, columns=ohe_cols, prefix=ohe_cols, prefix_sep='_', dummy_na=False)

# reorder so person_id and year_of_birth are first
cols = ['person_id', 'year_of_birth'] + [c for c in person_nodes_ohe.columns if c not in ('person_id', 'year_of_birth')]
person_nodes_ohe = person_nodes_ohe[cols]

# add indicator that these are person nodes for bipartite graph
person_nodes_ohe['person'] = True
person_nodes_ohe['condition'] = False

person_nodes_ohe.head()

,person_id,year_of_birth,gender_FEMALE,gender_MALE,race_American Indian or Alaska Native,race_Asian,race_Black or African American,race_Native Hawaiian or Other Pacific Islander,race_No matching concept,race_White,ethnicity_Hispanic or Latino,ethnicity_No matching concept,ethnicity_Not Hispanic or Latino,person,condition
0,115967179,1953,True,False,False,False,False,False,True,False,False,True,False,True,False
1,115967279,1969,False,True,False,True,False,False,False,False,False,False,True,True,False
2,115969631,1981,False,True,False,True,False,False,False,False,False,False,True,True,False
3,115972932,1982,False,True,False,False,False,False,True,False,False,False,True,True,False
4,115968415,1951,False,True,False,False,False,False,False,True,False,False,True,True,False


In [8]:
# write person node feature matrix to csv
person_nodes_ohe.to_csv("person_vertices.csv", index=False)

# 2. Prepare list of condition nodes

In [9]:
conditions = visits[['condition_concept_id']].drop_duplicates().reset_index(drop=True)
conditions.shape # 6376 unique conditions
# add concept name to conditions
conditions = conditions.merge(concepts[['concept_id', 'concept_name']], left_on='condition_concept_id', right_on='concept_id', how='left')
conditions.head()

,condition_concept_id,concept_id,concept_name
0,440370,440370.0,Nutritional marasmus
1,4233565,4233565.0,Severe protein-calorie malnutrition (Gomez: le...
2,436078,436078.0,Malnutrition of moderate degree (Gomez: 60 per...
3,4156515,4156515.0,Malnutrition (calorie)
4,435227,435227.0,Nutritional deficiency disorder


In [ ]:
# find exceedingly common conditions
# count unique person_id for each condition_concept_id
condition_counts = visits.groupby('condition_concept_id')['person_id'].nunique()
# join counts to conditions
conditions_counted = conditions.merge(condition_counts, on='condition_concept_id', how='left')
conditions_counted.rename(columns={'person_id': 'occurrences'}, inplace=True)
conditions_counted['frequency'] = conditions_counted['occurrences'] / len(people_filter)  # normalize
conditions_counted.sort_values(by='occurrences', ascending=False).head(20) # most common conditions
conditions_counted['frequency'].describe()  # distribution of condition frequencies
outlier_high = conditions_counted['frequency'].describe().loc['mean'] + (4 * conditions_counted['frequency'].describe().loc['std'])
outlier_high
# we can reasonably remove conditions that occur in more than 10% of people

np.float64(0.09538179439204507)

In [36]:
# filter very common conditions and very rare conditions
conditions_filtered = conditions_counted[conditions_counted['frequency'] <= 0.1]
conditions_filtered = conditions_filtered[conditions_filtered['occurrences'] >= 10].drop(columns='concept_id').reset_index(drop=True)
conditions_counted.shape # 6376 rows
conditions_filtered.shape # reduced to 1683 rows

(1683, 4)

In [38]:
conditions_filtered.head()
conditions_filtered.to_csv("condition_vertices.csv", index=False)

# 3. Define edge list, mapping person nodes to conditions they experienced

In [39]:
# remove repeat pairs of person/condition
visits_dedupe = visits.drop_duplicates(subset=['person_id', 'condition_concept_id']).reset_index(drop=True)
visits_dedupe.shape

(95409, 19)

In [ ]:
# remove conditions not in filtered set
visits_filtered = visits_dedupe[visits_dedupe['condition_concept_id'].isin(conditions_filtered['condition_concept_id'])]
visits_filtered.shape # reduced to 60,000 edges

(60776, 19)

In [41]:
# edge list will just be id columns
edges = visits_filtered[['person_id', 'condition_concept_id']].copy()
edges.head()
edges.to_csv("edge_list.csv", index=False)